<a href="https://colab.research.google.com/github/StrawEater/PracticasPDI3erBimestre/blob/main/Laboratorios/Laboratorio1_Pdi20251.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Laboratorio 1: Operadores Puntuales e Histograma

In [ ]:
import unittest
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, útil


## 1. **Operadores Puntuales**


Los Operadores Puntuales son funciónes matemáticas que mapean los niveles de gris de una imagen a otros niveles de gris, obteniendo una nueva imagen. Estos operadores trabajan de manera píxel a píxel, modificando los valores sin tener en cuenta su relación con otros píxeles en la imagen (es decir, no consideran su vecindad).
### ***Ejemplo de Operadores Puntuales***:
-   **Transformación lineal**: Una de las transformaciónes más simples es multiplicar cada nivel de gris por un factor constante $n$, como $x→x⋅n$, donde $x$ es el valor de gris de un píxel en la imagen. Esta es una transformación puntual simple que puede hacer que la imagen sea más brillante (si $n>1$) o más oscura (si $n<1$).
- **Transformaciónes no lineales**: Como las funciónes exponenciales o logarítmicas, que son útiles cuando se desea comprimir o expandir ciertos rangos de intensidad en una imagen.
    -   Ejemplo: $x=log⁡(x+1) \cdot x$ que oscurece las áreas brillantes y resalta las áreas oscuras.
-   **Escala de grises**: Otro tipo de transformación puntual podría ser simplemente reemplazar los niveles de gris de la imagen por un valor constante: $x→c$. Esto puede ser útil para crear una imagen binaria (blanco y negro), lo que es útil para técnicas como la segmentación.


In [ ]:
# Función que devuelve un gradiente (Imagen en escala de grises) de 256x256 en uint8
def gradiente():
  grandiante = np.arange(0,256).reshape(-1,256).astype(np.uint8)
  grandiante = np.tile(grandiante, (256, 1))
  return grandiante


In [ ]:
# Definimos una transformación puntual, de nivel de gris a nivel de gris
# Si modifican esta función, tenga cuidado de tener en cuenta overflowing del tipo de dato
# (Ver hoja de ayuda)
def T(x):
  if x > 100 and x < 190:
    return 20
  return x
# np.vectorize transforma nuestra función para ser correctamente aplicada
# elemento a elemento
vectorize_T = np.vectorize(T)
# Creamos un Gradiente simple 256 x 256
gradienteOriginal = gradiente()
# Aplicamos la transformación puntual en el gradiente
gradienteTransformado = vectorize_T(gradiente())
# Mostramos el resultado de la transformación
fig, axs = plt.subplots(1,3, figsize=(10,10))
axs[0].imshow(gradienteOriginal, cmap='gray', clim=(0,255))
axs[0].set_title("Imagen Original")
axs[0].axis("off")
# Mostramos la representacion de la Transformación sobre los niveles de gris
axs[1].plot(np.arange(0,256),vectorize_T(np.arange(0,256)))
axs[1].set_title("Transformación")
axs[1].set_xlim(0, 255)
axs[1].set_ylim(0, 255)
axs[1].set_aspect('equal')
axs[1].grid(True)
axs[2].imshow(gradienteTransformado, cmap='gray', clim=(0,255))
axs[2].set_title("Imagen Transformada")
axs[2].axis("off")
plt.show()


### 1. **Multiplicación**


Crear una familia de Transformaciónes Puntuales de la forma `x⋅n`, donde **x** es el nivel de gris original y **n** un numero natural.
Recuerden que la imagen resultante de aplicar la Transformación debe tener un formato y rango valido.


In [ ]:
# Deben definir una transformación puntual de nivel de gris a nivel de gris.
# Tengan en cuenta Overflowing del tipo de dato (Ver hoja de ayuda)
# Piensen en mover el tipo de dato a uno de mas precisión y despues útilizar np.clip
def multiplicación_puntual(x, n):


In [ ]:
class TestTransformación(unittest.TestCase):
    def test_identidad(self):
      #Si multiplicamos por 1, nos deberia devolver el mismo resultado
      x = np.array([0, 100, 200, 255], dtype=np.uint8)
      np.testing.assert_array_equal(multiplicación_puntual(x, 1), x, err_msg="Con n=1 la imagen debería permanecer igual")
    def test_multiplicación_sin_clipping(self):
      #Hacemos multiplicación normal sin llegar al clip
      x = np.array([0, 50, 127], dtype=np.uint8)
      esperado = np.array([0, 100, 254], dtype=np.uint8)
      np.testing.assert_array_equal(multiplicación_puntual(x, 2), esperado, err_msg="Multiplicación sin clipping falló para valores ≤127")
    def test_clipping_superior(self):
      #Debemos hacer clipping
      x = np.array([130, 200, 255], dtype=np.uint8)
      esperado = np.array([255, 255, 255], dtype=np.uint8)
      np.testing.assert_array_equal(multiplicación_puntual(x, 2), esperado, err_msg="El clipping superior (a 255) no se aplicó correctamente")
    def test_n_cero(self):
      #Debemos devolver 0
      x = np.array([10, 50, 200], dtype=np.uint8)
      esperado = np.zeros_like(x)
      np.testing.assert_array_equal(multiplicación_puntual(x, 0), esperado,err_msg="Con n=0 todos los píxeles deberían ser 0")
    def test_n_grande(self):
      #Debemos hacer que clip siga funciónando
      x = np.array([100], dtype=np.uint8)
      self.assertEqual(multiplicación_puntual(x, 1000), 255, "Con n=1000 el resultado debería ser 255 (clipping)")
    def test_tipo_salida(self):
      #Debemos devolver la salida correcta
      x = np.array([10, 20], dtype=np.uint8)
      self.assertEqual(multiplicación_puntual(x, 3).dtype, np.uint8, "El tipo de dato de salida debe ser np.uint8")
    def test_rango(self):
      #Deberia funcióna en una imagen
      x = np.random.randint(0, 256, size=(100, 100), dtype=np.uint8)
      res = multiplicación_puntual(x, 5)
      self.assertTrue(np.all(res >= 0) and np.all(res <= 255), "Hay valores fuera del rango [0, 255] en la salida")
    def test_escalar(self):
      #Si es un escalar, devuelve escalar
      self.assertEqual(multiplicación_puntual(100, 2), 200, "Multiplicación escalar con n=2 dio resultado incorrecto")
      self.assertEqual(multiplicación_puntual(200, 2), 255, "Clipping para escalar con n=2 y valor 200 debería dar 255")
    def test_overflow_previene(self):
      #Testear overflow
      self.assertEqual(multiplicación_puntual(200, 2), 255,"Overflow. La multiplicación de 200*2 en uint8 da 144, pero se espera 255. "
        "tenes que convertir a un tipo de mayor precisión (float64) antes de multiplicar.")
if __name__ == '__main__':
  suite = unittest.TestLoader().loadTestsFromTestCase(TestTransformación)
  resultado = unittest.TextTestRunner(verbosity=0).run(suite)
  if resultado.wasSuccessful():
    print("Todos los tests pasaron correctamente. La transformación funcióna bien.")
  else:
    print("Algunos tests fallaron. Mirá los mensajes de error arriba.")


In [ ]:
vectorize_T = np.vectorize(multiplicación_puntual)
# Creamos un Gradiente simple 256 x 256
gradienteOriginal = gradiente()
# Numero al Multiplicar
n = 2
# Aplicamos la transformación puntual en el gradiente
gradienteTransformado = vectorize_T(gradiente(), n)
# Mostramos el resultado de la transformación
fig, axs = plt.subplots(1,3, figsize=(10,10))
axs[0].imshow(gradienteOriginal, cmap='gray', clim=(0,255))
axs[0].set_title("Imagen Original")
axs[0].axis("off")
# Mostramos la representacion de la Transformación sobre los niveles de gris
axs[1].plot(np.arange(0,256),vectorize_T(np.arange(0,256), n))
axs[1].set_title(f"Transformación: x ⋅ {n}")
axs[1].set_xlim(0, 255)
axs[1].set_ylim(0, 255)
axs[1].set_aspect('equal')
axs[1].grid(True)
axs[2].imshow(gradienteTransformado, cmap='gray', clim=(0,255))
axs[2].set_title(f"Imagen Transformada (n={n})")
axs[2].axis("off")
plt.show()


### 2. **Invertir**

Crear una Transformaciónes Puntual que dada una imagen, devuelva su **negativo**. Comparar su resultado con la implementacion de **Scikit-Image**
Recuerden que la imagen resultante de aplicar la Transformación debe tener un formato y rango valido.


In [ ]:
# Definimos una transformación puntual, de nivel de gris a nivel de gris
def transformación_negativa(x):


In [ ]:
class TestNegativo(unittest.TestCase):
    def test_identidad_doble(self):
      #Si aplicamos negativo dos veces debe devolver la imagen original
      x = np.array([0, 50, 100, 200, 255], dtype=np.uint8)
      resultado = transformación_negativa(transformación_negativa(x))
      np.testing.assert_array_equal(resultado, x, err_msg="Aplicar negativo dos veces no devuelve la original")
    def test_valores_extremos(self):
      #Comprobamos que 0->255 y 255->0
      self.assertEqual(transformación_negativa(0), 255, "transformación_negativa(0) debería ser 255")
      self.assertEqual(transformación_negativa(255), 0, "transformación_negativa(255) debería ser 0")
    def test_valores_intermedios(self):
      #Comprueba que 100->155, 200->55 127 -> 128
      self.assertEqual(transformación_negativa(100), 155, "transformación_negativa(100) debería ser 155")
      self.assertEqual(transformación_negativa(200), 55, "transformación_negativa(200) debería ser 55")
      self.assertEqual(transformación_negativa(127), 128, "transformación_negativa(127) debería ser 128")
    def test_rango(self):
      #Verificamos que todos los valores de salida estén en [0,255] en una imagen random
      x = np.random.randint(0, 256, size=(50, 50), dtype=np.uint8)
      res = transformación_negativa(x)
      self.assertTrue(np.all(res >= 0) and np.all(res <= 255), "Hay valores fuera de rango")
    def test_tipo_dato(self):
      #La salida debe ser np.uint8
      x = np.array([10, 20], dtype=np.uint8)
      self.assertEqual(transformación_negativa(x).dtype, np.uint8, "El tipo de dato de salida debe ser np.uint8")
    def test_comparacion_skimage(self):
      #Comparamos nuestro negativo con el de skimage.útil.invert
      imagen = data.camera()
      propio = transformación_negativa(imagen)
      sk = útil.invert(imagen)
      np.testing.assert_array_equal(propio, sk, err_msg="Nuestro negativo difiere del de scikit-image")
    def test_escalar(self):
      #Aceptamos escalares y devolvemos escalares
      self.assertEqual(transformación_negativa(50), 205, "transformación_negativa(50) debería ser 205")
      self.assertEqual(transformación_negativa(0), 255, "transformación_negativa(0) debería ser 255")
if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestNegativo)
    resultado = unittest.TextTestRunner(verbosity=0).run(suite)
    if resultado.wasSuccessful():
        print("Todos los tests pasaron. El negativo funcióna correctamente.")
    else:
        print("Algunos tests fallaron. Revisá la implementación.")


In [ ]:
vectorize_T = np.vectorize(transformación_negativa)
# Cargamos una imagen de ejemplo
imagenOriginal = data.camera()
# Aplicamos la transformación puntual en la imagen
imagenInvertida = vectorize_T(imagenOriginal.copy())
imagenInvertidaSK = útil.invert(imagenOriginal.copy())
fig, axs = plt.subplots(2,2, figsize=(10,10))
axs = axs.ravel()
axs[0].imshow(imagenOriginal, cmap='gray', clim=(0,255))
axs[0].set_title("Imagen Original")
axs[0].axis("off")
# Mostramos la representacion de la Transformación sobre los niveles de gris
axs[1].plot(np.arange(0,256),vectorize_T(np.arange(0,256)))
axs[1].set_title("Transformación")
axs[1].set_xlim(0, 255)
axs[1].set_ylim(0, 255)
axs[1].set_aspect('equal')
axs[1].grid(True)
axs[2].imshow(imagenInvertidaSK, cmap='gray', clim=(0,255))
axs[2].set_title("Imagen Invertida SK")
axs[2].axis("off")
axs[3].imshow(imagenInvertida, cmap='gray', clim=(0,255))
axs[3].set_title("Su Imagen Invertida")
axs[3].axis("off")
plt.show()


### 3. **Threshold**


Implementar una función que dada una imagen y un valor de umbral, devuelva una **Imagen binarizada** (cada valor es estrictamente 0 o 255)


In [ ]:
#Creamos una función para crear una copia de la imagen binarizada
def BinarizarImagen(imagen, umbral):
  """
  Binariza una imagen en escala de grises según un umbral fijo.
    - Los píxeles con valor >= umbral se establecen a 255 (blanco).
    - Los píxeles con valor < umbral se establecen a 0 (negro).
  """


In [ ]:
class TestBinarizacion(unittest.TestCase):

  def test_umbral_bajo(self):
    #Con umbral = 0, todos los píxeles (>=0) deberian ser 255
    x = np.array([0, 100, 200, 255], dtype=np.uint8)
    esperado = np.array([255, 255, 255, 255], dtype=np.uint8)
    np.testing.assert_array_equal(BinarizarImagen(x, 0), esperado,err_msg="Con umbral 0 todos deberían ser 255")

  def test_umbral_alto(self):
    #Con umbral = 256 (o >255), todos los píxeles deben ser 0
    x = np.array([0, 100, 200, 255], dtype=np.uint8)
    esperado = np.array([0, 0, 0, 0], dtype=np.uint8)
    np.testing.assert_array_equal(BinarizarImagen(x, 256), esperado,err_msg="Con umbral >255 todos deberían ser 0")

  def test_umbral_medio(self):
    #Verificamos la separación correcta para un umbral intermedio
    x = np.array([0, 100, 128, 200, 255], dtype=np.uint8)
    umbral = 128
    esperado = np.array([0, 0, 255, 255, 255], dtype=np.uint8)
    np.testing.assert_array_equal(BinarizarImagen(x, umbral), esperado, err_msg="Fallo en la separación para umbral 128")

  def test_umbral_limite_inferior(self):
    #Con umbral = 1, solo el 0 debería ser 0, el resto debería 255
    x = np.array([0, 1, 2], dtype=np.uint8)
    esperado = np.array([0, 255, 255], dtype=np.uint8)
    np.testing.assert_array_equal(BinarizarImagen(x, 1), esperado, err_msg="Con umbral 1, solo el 0 debe ser 0")

  def test_umbral_limite_superior(self):
    #Con umbral = 255, solo el 255 debería ser 255, los demás 0
    x = np.array([0, 100, 254, 255], dtype=np.uint8)
    esperado = np.array([0, 0, 0, 255], dtype=np.uint8)
    np.testing.assert_array_equal(BinarizarImagen(x, 255), esperado, err_msg="Con umbral 255, solo el 255 debe ser 255")

  def test_tipo_salida(self):
    #La salida debe ser uint8
    x = np.array([10, 20], dtype=np.uint8)
    self.assertEqual(BinarizarImagen(x, 50).dtype, np.uint8, "El tipo de dato de salida debe ser np.uint8")

  def test_rango_salida(self):
    #Todos los valores de salida deben ser 0 o 255 en una imagen
    x = np.random.randint(0, 256, size=(100, 100), dtype=np.uint8)
    res = BinarizarImagen(x, 100)
    self.assertTrue(np.all((res == 0) | (res == 255)), "La salida contiene valores distintos de 0 o 255")

  def test_escalar(self):
    #La función debe aceptar un escalar como entrada
    self.assertEqual(BinarizarImagen(100, 50), 255)
    self.assertEqual(BinarizarImagen(100, 150), 0)

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestBinarizacion)
    resultado = unittest.TextTestRunner(verbosity=0).run(suite)

    if resultado.wasSuccessful():
        print("Todos los tests de binarización pasaron correctamente")
    else:
        print("Algunos tests fallaron. Revisa los mensajes de error")

In [ ]:
UMBRAL = 50
# Cargamos una imagen de ejemplo y su binarizacion
imagenOriginal = data.camera()
imagenBin = BinarizarImagen(imagenOriginal, UMBRAL)
fig, axs = plt.subplots(2, 2, figsize=(10, 8))
axs[0, 0].imshow(imagenOriginal, cmap='gray', vmin=0, vmax=255)
axs[0, 0].set_title("Imagen Original")
axs[0, 0].axis("off")
axs[0, 1].imshow(imagenBin, cmap='gray', vmin=0, vmax=255)
axs[0, 1].set_title(f"Imagen Binarizada (Umbral = {UMBRAL})")
axs[0, 1].axis("off")
#Mostramos el rango de valores actuales con el umbral
axs[1, 0].hist(imagenOriginal.ravel(), bins=256, range=(0, 256), color='black', histtype='step')
axs[1, 0].axvline(x=UMBRAL, color='r', linestyle='--', label=f'Umbral ({UMBRAL})')
axs[1, 0].set_title("Histograma Original")
axs[1, 0].set_xlim(0, 255)
axs[1, 0].legend()
#y como queda despues de la binarización
axs[1, 1].hist(imagenBin.ravel(), bins=256, range=(0, 256), color='black', histtype='step')
axs[1, 1].set_title("Histograma Binarizado (Solo 0 y 255)")
axs[1, 1].set_xlim(-5, 260)
plt.tight_layout()
plt.show()


## 2. **Histograma**

Un **histograma** es una herramienta poderosa en el procesamiento de imágenes que nos permite abstraer y describir el contenido de una imagen enfocándonos únicamente en los **niveles de gris**.
### ***¿Cómo funcióna un histograma?***
-   Un histograma se construye dividiendo los posibles valores de los niveles de gris (en imágenes en escala de grises) en intervalos llamados **bins**.
-   Los **bins** definen la resolución del histograma y nos dicen si estamos observando cada nivel de gris por separado o si agrupamos varios niveles cercanos.
### ***Ejemplo de Bins***:
-   Si tenemos un histograma con **256 bins**, estamos considerando cada nivel de gris individualmente, lo que significa que estamos contando cuántos píxeles tienen un valor de gris específico, del 0 al 255.
-   Si, en cambio, útilizamos solo **2 bins**, agruparemos los niveles de gris en dos categorías: uno para los píxeles con niveles de gris menores a 127 (valores oscuros) y otro para los niveles de gris mayores o iguales a 127 (valores claros).
De esta forma, el histograma con 2 bins proporcionará solo una visión muy simplificada de la imagen, mientras que uno con 256 bins proporcionará una visión detallada, donde cada nivel de gris se cuenta por separado.
### ***Visualización de un histograma***
Al analizar el histograma de una imagen, podemos obtener información clave como:
-   **Distribución de brillo**: Si la imagen tiene un histograma desplazado hacia la izquierda, significa que la imagen es más oscura. Si está hacia la derecha, será más brillante.
-   **Contraste**: Un histograma que cubre un rango amplio de niveles de gris indica una imagen con buen contraste, mientras que un histograma estrecho indica que los tonos de gris son muy similares entre sí, lo que da lugar a una imagen con bajo contraste.
-   **Histograma plano vs. sesgado**: Si el histograma está muy concentrado en un solo rango, puede indicar problemas de sobreexposición (histograma sesgado hacia los valores más altos) o subexposición (histograma sesgado hacia los valores más bajos).


### Crear Histograma

Crear una función que dada una imagen uint8 y un numero de bins, devuelve su histograma (un array de numpy con la cantidad de pixeles que le pertenecen a cada bin).
Por obvias razones no pueden útilizar `numpy.histogram`


In [ ]:
# Para que sea mas facil de pensar, pueden dividir la función en tres casos:
# bins == 1, bins == 256, el resto.
# Pueden útilizar la función np.linspace si les resulta útil.
def Histograma(imagen, bins):
  """
  Calcula el histograma de una imagen en escala de grises con un número especificado de bins/buckets.
  Parámetros:
      imagen : np.ndarray (uint8) - imagen de entrada
      bins : int - número de bins del histograma (entre 1 y 256)
  Retorna:
      hist : np.ndarray (int64) - frecuencias de cada bucket NO TIENE SENTIDO TENER UN HISTOGRAMA 3.5 PIXELES EN UN BUCKET POR EJEMPLO
  """
  if bins < 1:
    raise ValueError("El número de bins debe ser al menos 1")
  if bins > 256:
    raise ValueError("El número de bins no puede ser mayor a 256 para imágenes uint8")
  #Aplanamos para iterar sobre ellos
  pixeles = imagen.ravel()
  hist = np.zeros(bins, dtype=np.int64)
  if bins == 1:
  elif bins == 256:
  else:
    #Definimos los bordes de los buckets
    bordes_bins = np.linspace(0, 256, bins + 1)
  return hist


In [ ]:
class TestHistograma(unittest.TestCase):

  def setUp(self):
    #Mockeamos una imagen de prueba
    self.img = np.array([[0, 50, 100],
                          [150, 200, 255]], dtype=np.uint8)
    self.total_pixeles = 6

  def test_bins_invalidos(self):
    with self.assertRaises(ValueError):
        Histograma(self.img, 0)
    with self.assertRaises(ValueError):
        Histograma(self.img, -5)
    with self.assertRaises(ValueError):
        Histograma(self.img, 300)

  def test_suma_total(self):
    #La suma de las frecuencias deberia ser igual al número total de píxeles
    for bins in [1, 2, 10, 100, 256]:
      hist = Histograma(self.img, bins)
      self.assertEqual(np.sum(hist), self.total_pixeles,f"Suma incorrecta para bins={bins}")

  def test_bins_1(self):
    #Con un solo bucket, todos los píxeles deben caer en él
    hist = Histograma(self.img, 1)
    self.assertEqual(hist[0], self.total_pixeles, "Con bins=1 el único bucket debe contener todos los píxeles")
    self.assertEqual(len(hist), 1, "El histograma debe tener longitud 1")

  def test_bins_256(self):
    #Con 256 buckets, debe coincidir exactamente con np.bincount
    hist = Histograma(self.img, 256)
    esperado = np.bincount(self.img.ravel(), minlength=256)
    np.testing.assert_array_equal(hist, esperado,err_msg="El histograma con bins=256 no coincide con np.bincount")

  def test_tipo_de_dato(self):
    #El histograma debe ser de tipo entero (int64)
    for bins in [1, 10, 100, 256]:
      hist = Histograma(self.img, bins)
      self.assertTrue(np.issubdtype(hist.dtype, np.integer), f"Tipo de dato no entero para bins={bins} (obtenido {hist.dtype})")

  def test_sin_valores_negativos(self):
    #El histograma no debe contener valores negativos
    for bins in [1, 10, 100, 256]:
      hist = Histograma(self.img, bins)
      self.assertTrue(np.all(hist >= 0),f"Valores negativos encontrados para bins={bins}")

  def test_forma_correcta(self):
    #El tamaño del histograma deberia ser exactamente la cantidad bins
    for bins in [1, 2, 10, 100, 256]:
      hist = Histograma(self.img, bins)
      self.assertEqual(len(hist), bins, f"Longitud incorrecta: esperada {bins}, obtenida {len(hist)}")

  def test_comparacion_con_np_histogram(self):
    #Comparamos con np.histogram para bins generales usando los mismos bordes
    for bins in [2, 3, 5, 10, 20]:
      hist_nuestra = Histograma(self.img, bins)
      hist_np, _ = np.histogram(self.img.ravel(), bins=bins, range=(0, 256))
      np.testing.assert_array_equal(hist_nuestra, hist_np, err_msg=f"No coincide con np.histogram para bins={bins}")

  def test_imagen_grande(self):
    #Probamos con una imagen más grande (100x100)
    img_grande = np.random.randint(0, 256, size=(100, 100), dtype=np.uint8)
    for bins in [1, 10, 50, 100, 256]:
      hist = Histograma(img_grande, bins)
      self.assertEqual(np.sum(hist), 10000, f"Suma incorrecta para bins={bins} en imagen grande")
      self.assertEqual(len(hist), bins, f"Longitud incorrecta para bins={bins} en imagen grande")

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestHistograma)
    resultado = unittest.TextTestRunner(verbosity=0).run(suite)

    if resultado.wasSuccessful():
      print("Todos los tests del histograma pasaron correctamente")
    else:
      print("Algunos tests del histograma fallaron. Revisá los mensajes de error")

In [ ]:
BINS = 100

imagenOriginal = data.camera()
hist = Histograma(imagenOriginal, BINS)

# Mostramos el resultado
fig, axs = plt.subplots(1,3, figsize=(15,5))

axs[0].imshow(imagenOriginal, cmap='gray', clim=(0,255))
axs[0].set_title("Imagen Original")
axs[0].axis("off")

# Histograma de Referencia
# No es necesario que sea igual, ya que dependiendo de como manejen
# los puntos flotantes, puede cambiar ligeramente.
# Concentrense en que la silueta sea similar.
axs[1].set_title(f"Histograma de Referencia (b={BINS})")
axs[1].grid(True)
axs[1].hist(imagenOriginal.ravel(), bins=BINS, histtype='step', color='black')

# Histograma de ustedes
axs[2].set_title(f"Histograma Propio (b={BINS})")
axs[2].grid(True)
axs[2].bar(range(BINS), hist, width=1, edgecolor="black")

plt.tight_layout()
plt.show()